In [ ]:
import numpy as np
import matplotlib
matplotlib.use('module://ipykernel.pylab.backend_inline')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from skimage import data, color
from skimage.transform import resize

print('Starting model run')
# Step 1: Load sample images (digits dataset from skimage)
# We'll use simple images like 'camera' and 'coins' as two classes
images = [
    resize(color.rgb2gray(data.astronaut()), (64, 64)),  # Class 0
    resize(data.camera(), (64, 64)),                     # Class 1
    resize(data.coins(), (64, 64))                       # Class 2
]

labels = [0, 1, 2]  # Assign labels for recognition

# Step 2: Function to extract Fourier features
def extract_features(image):
    # Apply Fourier Transform
    f_transform = np.fft.fft2(image)
    f_shift = np.fft.fftshift(f_transform)

    # Magnitude spectrum
    magnitude = np.abs(f_shift)

    # Flatten and take only a few values (to keep it simple)
    features = magnitude.flatten()[:500]  # first 500 values
    return features

# Step 3: Build dataset
X = []
y = []
for i, img in enumerate(images):
    for _ in range(10):  # replicate each image 10 times
        noisy_img = img + 0.05 * np.random.randn(*img.shape)  # add small noise
        X.append(extract_features(noisy_img))
        y.append(labels[i])

X = np.array(X)
y = np.array(y)

# Step 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 5: Train a simple classifier (KNN)
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

# Step 6: Test accuracy
y_pred = model.predict(X_test)
print('Recognition Accuracy:', accuracy_score(y_test, y_pred))
print('X shape', X.shape, 'y shape', y.shape)
print('Labels', np.unique(y))

# Step 7: Visualize one image and its frequency spectrum
sample_img = images[0]
f_transform = np.fft.fft2(sample_img)
f_shift = np.fft.fftshift(f_transform)
magnitude = np.log(1 + np.abs(f_shift))  # log scale for visibility

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.title('Original Image')
plt.imshow(sample_img, cmap='gray')
plt.axis('off')

plt.subplot(1,2,2)
plt.title('Frequency Spectrum')
plt.imshow(magnitude, cmap='gray')
plt.axis('off')
plt.tight_layout()

plt.show()
print('Plot complete')
